# PrintVision AI - retrain on a leak-free split (Colab GPU)

1. Downloads dataset **v5** (2,007 images) from Roboflow.
2. Builds a **leak-free split**: every copy of a photo (`_aug`, `_original`) stays in one split.
3. Trains **YOLOv8s** (bigger than the original YOLOv8n) for up to 100 epochs.
4. Evaluates on the held-out test set and downloads `printvision_v2.pt` + metrics.

**Before running:** Runtime → Change runtime type → **T4 GPU**. In the left sidebar open **Secrets (key icon)**, add `ROBOFLOW_API_KEY`, and switch on notebook access. Then Runtime → **Run all**.

In [ ]:
%pip install -q ultralytics roboflow
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU')

In [ ]:
# ---- settings --------------------------------------------------------------
WORKSPACE, PROJECT, VERSION = 'abilesh-athimulam', 'my-first-project-krsyz', 5
MODEL    = 'yolov8s.pt'   # 'yolov8n.pt' = original size; 's' is ~3x bigger and usually more accurate
EPOCHS   = 100
PATIENCE = 25             # stop early if validation mAP stops improving
IMGSZ    = 640            # same as the original model and the app
BATCH    = 16

In [ ]:
# ---- download dataset (key from Colab Secrets, never written in the code) ----
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8', location='/content/raw_v5')

In [ ]:
# ---- leak-free split (same logic as scripts/make_clean_split.py) ------------
import re, shutil, yaml
from pathlib import Path

SPLITS = ('train', 'valid', 'test')

def source_id(filename):
    stem = filename.split('.rf.')[0]
    stem = re.sub(r'_(jpe?g|png)$', '', stem, flags=re.I)
    return re.sub(r'(_aug|_original)+$', '', stem)

def clean_split(src, dst):
    files = {s: sorted((src / s / 'images').iterdir()) for s in SPLITS}
    owner = {}
    for split in ('train', 'test', 'valid'):
        for img in files[split]:
            owner.setdefault(source_id(img.name), split)
    for split in SPLITS:
        for img in files[split]:
            target = owner[source_id(img.name)]
            label = src / split / 'labels' / (img.stem + '.txt')
            for f, kind in ((img, 'images'), (label, 'labels')):
                if f.exists():
                    out = dst / target / kind / f.name
                    out.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, out)
    cfg = yaml.safe_load((src / 'data.yaml').read_text())          # works whatever the key order
    (dst / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(dst), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
         'nc': cfg['nc'], 'names': cfg['names']}, sort_keys=False))
    return {s: len(list((dst / s / 'images').iterdir())) for s in SPLITS}

CLEAN = Path('/content/v5_clean')
if CLEAN.exists(): shutil.rmtree(CLEAN)
print(clean_split(Path('/content/raw_v5'), CLEAN))   # expect about train 1649 / valid 279 / test 79
DATA_YAML = CLEAN / 'data.yaml'
print(DATA_YAML.read_text())

In [ ]:
# ---- train -----------------------------------------------------------------
from ultralytics import YOLO

run = YOLO(MODEL).train(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    cos_lr=True,        # smooth learning-rate decay
    close_mosaic=15,    # last epochs without mosaic -> better on single real photos
    degrees=5, hsv_v=0.4, fliplr=0.5,   # mild rotation / brightness / flip
    seed=0, deterministic=True,
    project='/content/runs', name='printvision_v2', exist_ok=True,
)
NEW = Path(run.save_dir) / 'weights' / 'best.pt'
print('Best weights:', NEW)

In [ ]:
# ---- evaluate on the held-out TEST split ------------------------------------
import json, numpy as np

m = YOLO(str(NEW)).val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, conf=0.001, plots=True,
                       project='/content/runs', name='printvision_v2_test', exist_ok=True)
f1 = m.box.f1_curve.mean(0); i = int(np.argmax(f1))
names = m.names
result = {
    'model': MODEL, 'epochs_run': int(run.epoch + 1) if hasattr(run, 'epoch') else None,
    'test_images': len(list((CLEAN / 'test' / 'images').iterdir())),
    'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map),
    'precision': float(m.box.mp), 'recall': float(m.box.mr),
    'best_F1': float(f1[i]), 'best_conf': float(m.box.px[i]),
    'per_class_AP50': {names[c]: float(ap) for c, ap in zip(m.box.ap_class_index, m.box.ap50)},
}
print(json.dumps(result, indent=2))
Path('/content/printvision_v2_metrics.json').write_text(json.dumps(result, indent=2))

In [ ]:
# ---- download the results to your computer ---------------------------------
from google.colab import files
shutil.copy2(NEW, '/content/printvision_v2.pt')
shutil.make_archive('/content/printvision_v2_plots', 'zip', '/content/runs')
for f in ('printvision_v2.pt', 'printvision_v2_metrics.json', 'printvision_v2_plots.zip'):
    files.download('/content/' + f)